In [15]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [16]:
# Cargar dataset

df = pd.read_parquet("../data/grid_ocean/NOAA_encology.parquet")

In [17]:
# Convertir medición de microplásticos a categórica basada en rangos

bins = [0, 1, df['microplastics_measurement'].max()]  # ajusta según describe()
labels = ['low', 'high']

# Crear nueva columna con rangos
df['concentration_class_range'] = pd.cut(
    df['microplastics_measurement'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Revisar la distribución por rango
df['concentration_class_range'].value_counts()

concentration_class_range
low     11722
high     2433
Name: count, dtype: int64

In [18]:
features = ['temperature', 'salinity', 'chlorophyll', 'nitrate',
            'phosphate', 'oxygen_dissolved', 'oxygen_utilization']
target = 'concentration_class_range'  
X = df[features]
y = df[target]

In [19]:
# Dividir datos en entrenamiento y prueba 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar modelo de clasificación
model = RandomForestClassifier(n_estimators=200,
                               class_weight='balanced',
                               random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [20]:
# Evaluar el modelo
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

        high       0.65      0.83      0.73       489
         low       0.96      0.91      0.93      2342

    accuracy                           0.89      2831
   macro avg       0.81      0.87      0.83      2831
weighted avg       0.91      0.89      0.90      2831



In [21]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_pred)


array([[ 405,   84],
       [ 214, 2128]])

In [22]:
# Ajustar umbral de clasificación
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.6).astype(int)

In [23]:
from sklearn.metrics import roc_auc_score, average_precision_score

roc_auc_score(y_test, y_proba)

0.9365861069926074